# 🐾 Animal Sound Generator — Colab Training v4

**Path B: Diffusion direct generation from noise.** No VAE needed.

| Step | Model | Time |
|------|-------|------|
| 1 | Autoencoder (optional) | ~1.5 hrs |
| 2 | Diffusion UNet (120M) | ~3-4 hrs |
| 3 | Generate & listen | 5 sec |

### Before running:
1. Upload `animal_audio.tar.gz` to Google Drive root
2. Upload existing checkpoints to Drive (optional)
3. Runtime → L4 GPU

In [ ]:
# @title 1. Setup
from google.colab import drive
drive.mount('/content/drive')

!git clone https://github.com/grindydev/animal_sound_generator.git /content/animal_sound_generator
%cd /content/animal_sound_generator
!git pull

!pip install -q torch torchaudio torchvision --index-url https://download.pytorch.org/whl/cu121
!pip install -q numpy matplotlib pandas scikit-learn librosa soundfile tqdm

!mkdir -p models/diffusion_checkpoints/train
!mkdir -p models/autoencoder_checkpoints/train

import torch
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
print(f'PyTorch {torch.__version__} | CUDA: {torch.cuda.is_available()}')

In [ ]:
# @title 2. Load Data + Restore Checkpoints from Drive
import os, tarfile

# Data
LOCAL = '/content/animal_sound_generator/data'
TAR = '/content/drive/MyDrive/animal_audio.tar.gz'
if os.path.isdir(os.path.join(LOCAL, 'animal_audio')):
    print('✅ Data already loaded')
elif os.path.exists(TAR):
    print('📂 Extracting...')
    os.makedirs(LOCAL, exist_ok=True)
    with tarfile.open(TAR, 'r:gz') as tf:
        tf.extractall(path=LOCAL, filter='data')
    nested = os.path.join(LOCAL, 'data', 'animal_audio')
    if os.path.isdir(nested):
        !mv {nested} {LOCAL}/animal_audio && rmdir {LOCAL}/data
    print('✅ Done!')
else:
    print('❌ animal_audio.tar.gz not found in Drive root')

# Restore checkpoints
DRIVE = '/content/drive/MyDrive/animal_sound_generator/models'
if os.path.isdir(DRIVE):
    !cp -r {DRIVE}/* models/ 2>/dev/null
    !ls -lh models/*.pth 2>/dev/null || echo '(no .pth files yet)'
    print('✅ Checkpoints restored')

!ls data/animal_audio/

In [ ]:
# @title 3. (Optional) Train Autoencoder (~1.5 hrs)
# Skip if best_autoencoder_train.pth exists from Drive
!python src/vae/train_ae.py

In [ ]:
# @title 4. ⭐ Train Diffusion v10 — ESC-50 (150 epochs)\n# Download ESC-50 (640 clean animal sound clips)\n!wget -q https://github.com/karolpiczak/ESC-50/archive/refs/heads/master.zip -O /tmp/esc50.zip\n!unzip -qo /tmp/esc50.zip -d /tmp/\n!python src/scripts/setup_esc50.py --source /tmp/ESC-50-master/audio --target data/esc50\n# Train (150 epochs)\n!python src/diffusion/train.py

In [ ]:
# @title 5. Generate Audio
!python src/generate.py --label Dog --from-scratch --diffusion-steps 100

In [ ]:
# @title 6. (Optional) Train VAE (~1.5 hrs)
# Only needed for style transfer experiments, not generation
!python src/vae/finetune.py

In [ ]:
# @title 💾 Save All to Drive
DRIVE = '/content/drive/MyDrive/animal_sound_generator/models'
!mkdir -p {DRIVE}
import os
for f in ['best_autoencoder_train.pth', 'best_vae_finetune_train.pth',
          'best_audio_cnn_train.pth', 'diffusion_unet_train_best.pth',
          'hifigan_generator_train_best.pth']:
    path = f'models/{f}'
    if os.path.exists(path):
        !cp {path} {DRIVE}/
        print(f'  ✅ {f} ({os.path.getsize(path)/1e6:.0f} MB)')
for d in ['autoencoder_checkpoints', 'diffusion_checkpoints']:
    if os.path.isdir(f'models/{d}'):
        !cp -r models/{d} {DRIVE}/
        print(f'  ✅ {d}/')
print(f'\n📂 Saved to {DRIVE}/')

In [ ]:
# @title 🔁 Resume After Timeout
DRIVE_M = '/content/drive/MyDrive/animal_sound_generator/models'
if os.path.isdir(DRIVE_M):
    !cp -r {DRIVE_M}/* models/ 2>/dev/null
    print('✅ Checkpoints restored')
import tarfile
TAR = '/content/drive/MyDrive/animal_audio.tar.gz'
LOCAL = '/content/animal_sound_generator/data'
if not os.path.isdir(os.path.join(LOCAL, 'animal_audio')) and os.path.exists(TAR):
    os.makedirs(LOCAL, exist_ok=True)
    with tarfile.open(TAR, 'r:gz') as tf:
        tf.extractall(path=LOCAL, filter='data')
    nested = os.path.join(LOCAL, 'data', 'animal_audio')
    if os.path.isdir(nested):
        !mv {nested} {LOCAL}/animal_audio && rmdir {LOCAL}/data
    print('✅ Data re-extracted')
print('Ready — re-run training cell')


In [ ]:
# @title 🎧 Generate & Listen
# DDIM (fast, 100 steps) or DDPM (slower, more diverse)
!python src/generate.py --label Dog --from-scratch --diffusion-steps 100

from IPython.display import Audio, display
import glob
wavs = sorted(glob.glob('generated_audio/*.wav'))
if wavs:
    for w in wavs[-3:]:
        display(Audio(w, rate=22050))
        print(f'🔊 {w}')
else:
    print('No audio — train diffusion first')